In [1]:
#install.packages("baseballr")
#install.packages("dplyr")
#install.packages("readr")
#install.packages("lubridate")
#install.packages("tidyverse")
#install.packages("duckdb")
install.packages("purrr")
library(purrr)
library(duckdb)
library(tidyverse)
library(baseballr)
library(dplyr)
library(readr)
library(lubridate)


The downloaded binary packages are in
	/var/folders/4k/3pfkn1jn039bp8wy8410pn3m0000gn/T//RtmpA4ULcv/downloaded_packages


Loading required package: DBI

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


## WHEN READING THIS FILE, KEEP IN MIND THAT THESE ARE INCREMENTAL STEPS ALONG THE DATA LOADING PROCESS THAT IS PRIMARILY CONDUCTED IN project.ipynb. THIS IS NOT INTENDED TO BE ONE STRAIGHT WORKFLOW

In [2]:
GBS <- read_csv("/Users/owendrummond/Documents/python_projects/Capstone/GameBoxScores_PostBullpen.csv")

Rows: 17906 Columns: 245
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (72): DayOfWeek, AwayTeam, AwayLeague, HomeTeam, HomeLeague, Day/Night...
dbl  (172): GAMEID, Unnamed: 0, AwayGameNum, HomeGameNum, AwayScore, HomeSco...
date   (1): Date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [3]:
GBSHitting <- read_csv("/Users/owendrummond/Documents/python_projects/Capstone/GBSHitting.csv")

Rows: 17906 Columns: 829
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (72): DayOfWeek, AwayTeam, AwayLeague, HomeTeam, HomeLeague, Day/Night...
dbl  (756): GAMEID, Unnamed: 0, AwayGameNum, HomeGameNum, AwayScore, HomeSco...
date   (1): Date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [137]:
# Unique pitcher season and ID combinations to be used in game log data gathering process.

unique_pitcher_seasons <- GBS %>%
  select(Year, HomePitcherIDFG, AwayPitcherIDFG) %>%
  pivot_longer(
    cols = c(HomePitcherIDFG, AwayPitcherIDFG), 
    names_to = "Side", 
    values_to = "PitcherID"
  ) %>%
  # Keep only one row per Pitcher/Season
  distinct(PitcherID, Year)

# View the result
print(unique_pitcher_seasons)

# A tibble: 2,890 × 2
   PitcherID  Year
       <dbl> <dbl>
 1      8782  2019
 2     10603  2019
 3     19427  2019
 4     16400  2019
 5     15440  2019
 6     10190  2019
 7      5401  2019
 8     14527  2019
 9      4676  2019
10     11486  2019
# ℹ 2,880 more rows


In [4]:
write_csv(unique_pitcher_seasons, "unique_pitcher_seasons.csv")

ERROR: Error: object 'unique_pitcher_seasons' not found


In [13]:
unique_pitcher_seasons_temp <- read_csv('/Users/owendrummond/Documents/python_projects/Capstone/unique_pitcher_seasons.csv')
player_dim <- read_csv('/Users/owendrummond/Documents/python_projects/Capstone/player_dim_live.csv')

Rows: 2890 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (2): PitcherID, Year

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 2522 Columns: 6
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (5): player_name, retro_id, bref_id, fg_id, Team
dbl (1): mlb_id

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [14]:
unique_pitcher_seasons_2026 <- unique_pitcher_seasons_temp %>%
  filter(Year >= 2023) %>%
  select(PitcherID) %>%
  distinct() %>%
  mutate(
    PitcherID = as.character(PitcherID),
    Year = 2026
  )

# 2. Get fg_id, ensure it is character, and rename
new_pitcher_ids <- player_dim %>%
  filter(is.na(retro_id) | retro_id == "") %>%
  select(PitcherID = fg_id) %>%
  mutate(
    PitcherID = as.character(PitcherID),
    Year = 2026
  )

In [17]:
unique_pitcher_seasons_2026 <- bind_rows(unique_pitcher_seasons_2026, new_pitcher_ids)

In [18]:
fg_pitcher_game_logs(playerid = unique_pitcher_seasons_2026$PitcherID[1],year = unique_pitcher_seasons_2026$Year[1])

PlayerName,playerid,Date,Opp,teamid,season,Team,HomeAway,Age,W,⋯,Events,EV,LA,Barrels,Barrel%,maxEV,HardHit,HardHit%,gamedate,dh
<chr>,<int>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<int>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>
Cal Quantrill,19312,2026-04-18,@SEA,13,2026,TEX,A,31,0,⋯,6,99.63333,26.00000,2,0.3333333,106.7,4,0.6666667,2026-04-18,0
Cal Quantrill,19312,2026-04-16,@ATH,13,2026,TEX,A,31,1,⋯,3,84.90000,18.33333,0,0.0000000,103.7,1,0.3333333,2026-04-16,0


In [144]:
fg_pitcher_game_logs(playerid = 8782,year = 2019)

Request failed [403]. Retrying in 1 seconds...

Request failed [403]. Retrying in 3 seconds...

2026-04-14 22:34:38.922271: Invalid arguments or no pitcher game log data available!



ERROR: Error in fg_pitcher_game_logs(playerid = 8782, year = 2019): object 'payload' not found


## OG Pitcher Run (Does not include 2026)

#### Avoid running on your own, this will take multiple hours

In [151]:
# Because this function was taking so long to execute, and because I was continually getting thrown HTTP request blockers, I had some some A.I. enhancements
# to avoid these blockages such as error detection and periodic sleeps. The idea for the function however, was my own.

checkpoint_file <- "master_logs_checkpoint.rds"

if (file.exists(checkpoint_file)) {
  all_logs_list <- readRDS(checkpoint_file)
  # Identify which ID/Year combos we already have
  completed <- bind_rows(all_logs_list) %>%
    distinct(PitcherID, Year)
  
  # Filter our queue to only include those NOT in the completed list
  queue <- unique_pitcher_seasons %>%
    anti_join(completed, by = c("PitcherID" = "PitcherID", "Year" = "Year"))
  
  message(paste("Resuming... skipping", nrow(completed), "already downloaded combos."))
} else {
  all_logs_list <- list()
  queue <- unique_pitcher_seasons
}

# THE LOOP
for(i in 1:nrow(queue)) {
  p_id <- queue$PitcherID[i]
  p_year <- queue$Year[i]
  
  message(paste0("Processing (", i, "/", nrow(queue), "): ID ", p_id, " Year ", p_year))
  
  # Try to get the log
  log <- tryCatch({
    fg_pitcher_game_logs(playerid = p_id, year = p_year)
  }, error = function(e) {
    if (grepl("403", e$message) || grepl("payload", e$message)) {
      message("!!! API Blocker/403 detected. Sleeping for 2 minutes to cool down...")
      Sys.sleep(120) 
      return(NULL)
    }
    return(NULL)
  })
  
  # DATA VALIDATION & SAVING
  if(!is.null(log) && nrow(log) > 0) {
    log$PitcherID <- p_id
    log$Year <- p_year
    all_logs_list[[length(all_logs_list) + 1]] <- log
  }
  
  # Save a checkpoint every 10 pitchers so you don't lose work
  if (i %% 10 == 0) {
    saveRDS(all_logs_list, checkpoint_file)
    message("--- Checkpoint saved.")
  }
  
  # VARIABLE SLEEP
  # Random sleep between 1.5 and 3.5 seconds to avoid detection
  Sys.sleep(runif(1, 1.5, 3.5))
}

master_pitcher_logs <- bind_rows(all_logs_list)
saveRDS(master_pitcher_logs, "master_pitcher_logs_final.rds")
message("Process Complete! All logs saved to master_pitcher_logs_final.rds")

Processing (1/2890): ID 8782 Year 2019

Processing (2/2890): ID 10603 Year 2019

Processing (3/2890): ID 19427 Year 2019

Processing (4/2890): ID 16400 Year 2019

Processing (5/2890): ID 15440 Year 2019

Processing (6/2890): ID 10190 Year 2019

Processing (7/2890): ID 5401 Year 2019

Processing (8/2890): ID 14527 Year 2019

Processing (9/2890): ID 4676 Year 2019

Processing (10/2890): ID 11486 Year 2019

--- Checkpoint saved.

Processing (11/2890): ID 16918 Year 2019

Processing (12/2890): ID 18383 Year 2019

Processing (13/2890): ID 13074 Year 2019

Processing (14/2890): ID 19309 Year 2019

Processing (15/2890): ID 4806 Year 2019

Processing (16/2890): ID 9323 Year 2019

Processing (17/2890): ID 14078 Year 2019

Processing (18/2890): ID 12970 Year 2019

Processing (19/2890): ID 12699 Year 2019

Processing (20/2890): ID 14288 Year 2019

--- Checkpoint saved.

Processing (21/2890): ID 8779 Year 2019

Processing (22/2890): ID 10021 Year 2019

Processing (23/2890): ID 7410 Year 2019

Proc

In [153]:
write_csv(master_pitcher_logs, "SPGameLogFact.csv")

## 2026 Pitcher Run. Same logic as before, but for 2026 data.

In [19]:
# 1. SETUP & RESUME LOGIC
# Updated file name to avoid overwriting your previous master logs
checkpoint_2026 <- "pitcher_logs_2026_checkpoint.rds"

if (file.exists(checkpoint_2026)) {
  logs_list_2026 <- readRDS(checkpoint_2026)
  
  # Identify already processed combos
  completed_2026 <- bind_rows(logs_list_2026) %>%
    distinct(PitcherID, Year) %>%
    mutate(PitcherID = as.character(PitcherID)) # Safety check for type matching
  
  # Filter the 2026 queue (anti_join removes what we already have)
  queue_2026 <- unique_pitcher_seasons_2026 %>%
    mutate(PitcherID = as.character(PitcherID)) %>%
    anti_join(completed_2026, by = c("PitcherID", "Year"))
  
  message(paste("Resuming 2026 pull... skipping", nrow(completed_2026), "completed combos."))
} else {
  logs_list_2026 <- list()
  queue_2026 <- unique_pitcher_seasons_2026
}

# 2. THE LOOP
if(nrow(queue_2026) > 0) {
  for(i in 1:nrow(queue_2026)) {
    curr_id <- queue_2026$PitcherID[i]
    curr_year <- queue_2026$Year[i]
    
    message(paste0("2026 Task (", i, "/", nrow(queue_2026), "): ID ", curr_id, " Year ", curr_year))
    
    # Try to fetch logs from FanGraphs
    # Note: For the 40 newly added fg_ids with no retro_id, 
    # we are still using their fg_id as the playerid.
    log_data <- tryCatch({
      fg_pitcher_game_logs(playerid = curr_id, year = curr_year)
    }, error = function(e) {
      if (grepl("403", e$message) || grepl("payload", e$message)) {
        message("!!! API Limit Hit. Cooling down for 120 seconds...")
        Sys.sleep(120) 
        return(NULL)
      }
      return(NULL)
    })
    
    # 3. VALIDATION & STORAGE
    if(!is.null(log_data) && nrow(log_data) > 0) {
      log_data$PitcherID <- curr_id
      log_data$Year <- curr_year
      logs_list_2026[[length(logs_list_2026) + 1]] <- log_data
    }
    
    # Checkpoint every 10 pitchers
    if (i %% 10 == 0) {
      saveRDS(logs_list_2026, checkpoint_2026)
      message("--- 2026 Checkpoint saved.")
    }
    
    # Variable sleep to mimic human browsing and avoid IP bans
    Sys.sleep(runif(1, 1.8, 3.8))
  }
}

# 4. FINAL ASSEMBLY
pitcher_logs_2026_final <- bind_rows(logs_list_2026)
saveRDS(pitcher_logs_2026_final, "pitcher_logs_2026_final.rds")
message("2026 Process Complete! File saved as pitcher_logs_2026_final.rds")

2026 Task (1/671): ID 19312 Year 2026

2026 Task (2/671): ID 16256 Year 2026

2026 Task (3/671): ID 27758 Year 2026

2026 Task (4/671): ID 15454 Year 2026

2026 Task (5/671): ID 22201 Year 2026

2026 Task (6/671): ID 15440 Year 2026

2026 Task (7/671): ID 27694 Year 2026

2026 Task (8/671): ID 31827 Year 2026

2026-04-24 02:04:34.27841: Invalid arguments or no pitcher game log data available!

2026 Task (9/671): ID 19899 Year 2026

2026-04-24 02:04:36.647829: Invalid arguments or no pitcher game log data available!

2026 Task (10/671): ID 27646 Year 2026

2026-04-24 02:04:39.046225: Invalid arguments or no pitcher game log data available!

--- 2026 Checkpoint saved.

2026 Task (11/671): ID 27470 Year 2026

2026 Task (12/671): ID 30055 Year 2026

2026 Task (13/671): ID 22264 Year 2026

2026 Task (14/671): ID 31843 Year 2026

2026 Task (15/671): ID 16944 Year 2026

2026 Task (16/671): ID 27782 Year 2026

2026 Task (17/671): ID 27487 Year 2026

2026-04-24 02:05:00.490103: Invalid argument

In [21]:
write_csv(pitcher_logs_2026_final, "StartingPGameLogFact2026.csv")

In [4]:
# Compile a tibble of unique combinations of batter IDs and year for game log data gathering.

unique_hitter_seasons <- GBSHitting %>%
  select(Year, HomeBatter1IDFG, HomeBatter2IDFG, HomeBatter3IDFG, HomeBatter4IDFG, HomeBatter5IDFG, HomeBatter6IDFG, HomeBatter7IDFG, HomeBatter8IDFG, HomeBatter9IDFG,
   AwayBatter1IDFG, AwayBatter2IDFG, AwayBatter3IDFG, AwayBatter4IDFG, AwayBatter5IDFG, AwayBatter6IDFG, AwayBatter7IDFG, AwayBatter8IDFG, AwayBatter9IDFG) %>%
  pivot_longer(
    cols = c(HomeBatter1IDFG, HomeBatter2IDFG, HomeBatter3IDFG, HomeBatter4IDFG, HomeBatter5IDFG, HomeBatter6IDFG, HomeBatter7IDFG, HomeBatter8IDFG, HomeBatter9IDFG,
   AwayBatter1IDFG, AwayBatter2IDFG, AwayBatter3IDFG, AwayBatter4IDFG, AwayBatter5IDFG, AwayBatter6IDFG, AwayBatter7IDFG, AwayBatter8IDFG, AwayBatter9IDFG), 
    names_to = "Side", 
    values_to = "BatterID"
  ) %>%
  # Keep only one row per Pitcher/Season
  distinct(BatterID, Year)

# View the result
print(unique_hitter_seasons)

# A tibble: 5,866 × 2
   BatterID  Year
      <dbl> <dbl>
 1    22266  2024
 2    22275  2024
 3    19913  2024
 4    24703  2024
 5    26151  2024
 6    19608  2024
 7    20536  2024
 8    16947  2024
 9    18551  2024
10    18568  2024
# ℹ 5,856 more rows


In [7]:
write_csv(unique_hitter_seasons, "uhs.csv")

In [20]:
unique_hitter_seasons_temp <- read_csv('/Users/owendrummond/Documents/python_projects/Capstone/uhs.csv')

Rows: 5866 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (2): BatterID, Year

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [22]:
unique_hitter_seasons_2026 <- unique_hitter_seasons_temp %>%
  filter(Year >= 2023) %>%
  select(BatterID) %>%
  distinct() %>%
  mutate(
    BatterID = as.character(BatterID),
    Year = 2026
  )

# 2. Get fg_id, ensure it is character, and rename
new_batter_ids <- player_dim %>%
  filter(is.na(retro_id) | retro_id == "") %>%
  select(BatterID = fg_id) %>%
  mutate(
    BatterID = as.character(BatterID),
    Year = 2026
  )

In [23]:
unique_hitter_seasons_2026 <- bind_rows(unique_hitter_seasons_2026, new_batter_ids)

In [25]:
fg_batter_game_logs(playerid = unique_hitter_seasons_2026$BatterID[1],year = unique_hitter_seasons_2026$Year[1])

PlayerName,playerid,Date,Team,Opp,season,Age,BatOrder,Pos,G,⋯,Events,EV,LA,Barrels,Barrel%,maxEV,HardHit,HardHit%,gamedate,dh
<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>
Xavier Edwards,22266,2026-04-22,MIA,STL,2026,26,2,DH,1,⋯,4,94.02500,-4.0000000,0,0.0000000,98.4,3,0.7500000,2026-04-22,0
Xavier Edwards,22266,2026-04-21,MIA,STL,2026,26,4,2B,1,⋯,4,92.65000,27.7500000,0,0.0000000,99.2,2,0.5000000,2026-04-21,0
Xavier Edwards,22266,2026-04-20,MIA,STL,2026,26,4,2B,1,⋯,2,79.75000,-12.0000000,1,0.5000000,99.2,1,0.5000000,2026-04-20,0
Xavier Edwards,22266,2026-04-19,MIA,MIL,2026,26,2,2B,1,⋯,2,89.95000,-6.0000000,0,0.0000000,91.8,0,0.0000000,2026-04-19,0
Xavier Edwards,22266,2026-04-18,MIA,MIL,2026,26,2,2B,1,⋯,3,91.43333,22.6666667,0,0.0000000,94.5,0,0.0000000,2026-04-18,0
Xavier Edwards,22266,2026-04-17,MIA,MIL,2026,26,2,2B,1,⋯,4,97.15000,32.0000000,2,0.5000000,103.7,3,0.7500000,2026-04-17,0
Xavier Edwards,22266,2026-04-15,MIA,@ATL,2026,26,2,2B,1,⋯,4,88.40000,-9.2500000,0,0.0000000,101.7,2,0.5000000,2026-04-15,0
Xavier Edwards,22266,2026-04-14,MIA,@ATL,2026,26,2,2B,1,⋯,3,79.56667,-5.6666667,0,0.0000000,95.7,1,0.3333333,2026-04-14,0
Xavier Edwards,22266,2026-04-13,MIA,@ATL,2026,26,2,2B,1,⋯,3,95.10000,13.0000000,1,0.3333333,98.2,2,0.6666667,2026-04-13,0


# OG Hitter Run

In [10]:
# Because this function was taking so long to execute, and because I was continually getting thrown HTTP request blockers, I had some some A.I. enhancements
# to avoid these blockages such as error detection and periodic sleeps. The idea for the function however, was my own.

# SETUP RESUME LOGIC & HITTER-SPECIFIC FILENAMES
checkpoint_file <- "master_hitter_logs_checkpoint.rds"

if (file.exists(checkpoint_file)) {
  all_hitter_logs_list <- readRDS(checkpoint_file)
  # Identify which ID/Year combos we already have
  completed <- bind_rows(all_hitter_logs_list) %>%
    distinct(BatterID, Year)
  
  # Filter our queue to only include those NOT in the completed list
  queue <- unique_hitter_seasons %>%
    anti_join(completed, by = c("BatterID" = "BatterID", "Year" = "Year"))
  
  message(paste("Resuming... skipping", nrow(completed), "already downloaded hitter combos."))
} else {
  all_hitter_logs_list <- list()
  queue <- unique_hitter_seasons
}

# THE LOOP
for(i in 1:nrow(queue)) {
  b_id <- queue$BatterID[i]
  b_year <- queue$Year[i]
  
  message(paste0("Processing Hitter (", i, "/", nrow(queue), "): ID ", b_id, " Year ", b_year))
  
  # Try to get the batter log
  log <- tryCatch({
    # Updated to hitter-specific function
    fg_batter_game_logs(playerid = b_id, year = b_year)
  }, error = function(e) {
    if (grepl("403", e$message) || grepl("payload", e$message)) {
      message("!!! API Blocker/403 detected. Sleeping for 2 minutes to cool down...")
      Sys.sleep(120) 
      return(NULL)
    }
    return(NULL)
  })
  
  # DATA VALIDATION & SAVING
  if(!is.null(log) && nrow(log) > 0) {
    # Ensure columns match join keys
    log$BatterID <- b_id
    log$Year <- b_year
    all_hitter_logs_list[[length(all_hitter_logs_list) + 1]] <- log
  }
  
  # Save a checkpoint every 10 hitters
  if (i %% 10 == 0) {
    saveRDS(all_hitter_logs_list, checkpoint_file)
    message("--- Hitter checkpoint saved.")
  }
  
  # VARIABLE SLEEP
  # Slightly longer random sleep is safer for larger hitter batches
  Sys.sleep(runif(1, 1.8, 4.0))
}

master_hitter_logs <- bind_rows(all_hitter_logs_list)
saveRDS(master_hitter_logs, "master_hitter_logs_final.rds")
message("Process Complete! All hitter logs saved to master_hitter_logs_final.rds")

Resuming... skipping 5370 already downloaded hitter combos.

Processing Hitter (1/496): ID 21622 Year 2025

Processing Hitter (2/496): ID 25805 Year 2025

Processing Hitter (3/496): ID 18821 Year 2025

Processing Hitter (4/496): ID 23968 Year 2025

Processing Hitter (5/496): ID 24770 Year 2025

Processing Hitter (6/496): ID 24782 Year 2025

Processing Hitter (7/496): ID 22165 Year 2024

Processing Hitter (8/496): ID 22566 Year 2025

Processing Hitter (9/496): ID 27784 Year 2024

Processing Hitter (10/496): ID 17452 Year 2025

--- Hitter checkpoint saved.

Processing Hitter (11/496): ID 20538 Year 2025

Processing Hitter (12/496): ID 31793 Year 2025

Processing Hitter (13/496): ID 24605 Year 2024

Processing Hitter (14/496): ID 12147 Year 2025

Processing Hitter (15/496): ID 16686 Year 2024

Processing Hitter (16/496): ID 22274 Year 2025

Processing Hitter (17/496): ID 22707 Year 2025

Processing Hitter (18/496): ID 8610 Year 2019

Processing Hitter (19/496): ID 16291 Year 2019

Process

# 2026 Hitter Run, same logic just for 2026 data.

In [26]:
# 1. SETUP RESUME LOGIC & 2026-SPECIFIC FILENAMES
checkpoint_hitter_2026 <- "hitter_logs_2026_checkpoint.rds"

if (file.exists(checkpoint_hitter_2026)) {
  hitter_logs_list_2026 <- readRDS(checkpoint_hitter_2026)
  
  # Identify which ID/Year combos we already have
  # Converting to character to avoid the double/character type mismatch error
  completed_hitters_2026 <- bind_rows(hitter_logs_list_2026) %>%
    distinct(BatterID, Year) %>%
    mutate(BatterID = as.character(BatterID))
  
  # Filter our 2026 queue
  queue_hitter_2026 <- unique_hitter_seasons_2026 %>%
    mutate(BatterID = as.character(BatterID)) %>%
    anti_join(completed_hitters_2026, by = c("BatterID", "Year"))
  
  message(paste("Resuming 2026 Hitter pull... skipping", nrow(completed_hitters_2026), "already processed."))
} else {
  hitter_logs_list_2026 <- list()
  queue_hitter_2026 <- unique_hitter_seasons_2026
}

# 2. THE LOOP
if(nrow(queue_hitter_2026) > 0) {
  for(i in 1:nrow(queue_hitter_2026)) {
    curr_b_id <- queue_hitter_2026$BatterID[i]
    curr_b_year <- queue_hitter_2026$Year[i]
    
    message(paste0("2026 Hitter Task (", i, "/", nrow(queue_hitter_2026), "): ID ", curr_b_id, " Year ", curr_b_year))
    
    # Try to fetch hitter logs from FanGraphs
    log_hitter_data <- tryCatch({
      fg_batter_game_logs(playerid = curr_b_id, year = curr_b_year)
    }, error = function(e) {
      if (grepl("403", e$message) || grepl("payload", e$message)) {
        message("!!! API Blocker detected. Cooling down for 120 seconds...")
        Sys.sleep(120) 
        return(NULL)
      }
      return(NULL)
    })
    
    # 3. DATA VALIDATION & STORAGE
    if(!is.null(log_hitter_data) && nrow(log_hitter_data) > 0) {
      log_hitter_data$BatterID <- curr_b_id
      log_hitter_data$Year <- curr_b_year
      hitter_logs_list_2026[[length(hitter_logs_list_2026) + 1]] <- log_hitter_data
    }
    
    # Checkpoint every 10 hitters
    if (i %% 10 == 0) {
      saveRDS(hitter_logs_list_2026, checkpoint_hitter_2026)
      message("--- 2026 Hitter checkpoint saved.")
    }
    
    # Random sleep to avoid rate limiting (slightly more conservative for hitters)
    Sys.sleep(runif(1, 2.0, 4.5))
  }
}

# 4. FINAL ASSEMBLY
hitter_logs_2026_final <- bind_rows(hitter_logs_list_2026)
saveRDS(hitter_logs_2026_final, "hitter_logs_2026_final.rds")
message("2026 Process Complete! File saved as hitter_logs_2026_final.rds")

2026 Hitter Task (1/929): ID 22266 Year 2026

2026 Hitter Task (2/929): ID 22275 Year 2026

2026 Hitter Task (3/929): ID 19913 Year 2026

2026 Hitter Task (4/929): ID 24703 Year 2026

2026-04-24 02:48:25.605648: Invalid arguments or no batter game logs data available!

2026 Hitter Task (5/929): ID 26151 Year 2026

2026 Hitter Task (6/929): ID 19608 Year 2026

2026 Hitter Task (7/929): ID 20536 Year 2026

2026-04-24 02:48:33.473598: Invalid arguments or no batter game logs data available!

2026 Hitter Task (8/929): ID 16947 Year 2026

2026 Hitter Task (9/929): ID 18551 Year 2026

2026-04-24 02:48:40.79313: Invalid arguments or no batter game logs data available!

2026 Hitter Task (10/929): ID 18568 Year 2026

--- 2026 Hitter checkpoint saved.

2026 Hitter Task (11/929): ID 10815 Year 2026

2026-04-24 02:48:46.41686: Invalid arguments or no batter game logs data available!

2026 Hitter Task (12/929): ID 18036 Year 2026

2026 Hitter Task (13/929): ID 11493 Year 2026

2026 Hitter Task (14/

In [12]:
write_csv(master_hitter_logs, "BatterGameLogFact.csv")

In [28]:
write_csv(hitter_logs_2026_final, "BatterGameLogFact2026.csv")

# Bullpen

In [111]:
# For each possible year and month, return pitcher data for bullpens

years <- c(2017,2018,2019,2020,2021,2022,2023,2024,2025)
months <- c(3,5,6,7,8,9)

results_list <- list()
counter <- 1

for (yr in years) {
  for (mon in months) {
    
    if (mon == 3){
        end_month <- 4
        first_day <- make_date(yr, mon, 1)
        first_day_following <- make_date(yr, end_month, 1)
        last_day  <- ceiling_date(first_day_following, "month") - days(1)
        month_table <- 4
    }
    else{
    first_day <- make_date(yr, mon, 1)
    last_day  <- ceiling_date(first_day, "month") - days(1)
    month_table <- mon
    }

    stats <- bref_daily_pitcher(t1 = as.character(first_day), t2 = as.character(last_day))

    if(!is.null(stats) && nrow(stats) > 0) {
      stats <- stats %>% mutate(query_year = yr, query_month = month_table)

      results_list[[counter]] <- stats
      counter <- counter + 1
    }

    Sys.sleep(3)

    #print(first_day)
    #print(last_day)
  }}

2026-04-12 16:29:31.03516: Invalid arguments or no daily pitcher data available!

2026-04-12 16:29:36.161138: Invalid arguments or no daily pitcher data available!

2026-04-12 16:29:41.247068: Invalid arguments or no daily pitcher data available!



In [118]:
# Filter down the pitching data into SP and RP

bref_monthly_pitching <- bind_rows(results_list)

bref_monthly_sp <- bref_monthly_pitching %>% filter(GS >= 1)
bref_monthly_rp <- bref_monthly_pitching %>% mutate(GS = as.numeric(GS), G = as.numeric(G)) %>% filter(GS == 0 | (GS < (G/2)))

In [120]:
write_csv(bref_monthly_sp, "bref_monthly_sp.csv")
write_csv(bref_monthly_rp, "bref_monthly_rp.csv")

# Fangraphs Pitching Fact

In [46]:
# Retrieve yearly FangGraphs pitching stats from 2015 to 2025

fg_pitching_fact15 <- fg_pitcher_leaders(startseason = 2015, endseason = 2015)
fg_pitching_fact16 <- fg_pitcher_leaders(startseason = 2016, endseason = 2016)
fg_pitching_fact17 <- fg_pitcher_leaders(startseason = 2017, endseason = 2017)
fg_pitching_fact18 <- fg_pitcher_leaders(startseason = 2018, endseason = 2018)
fg_pitching_fact19 <- fg_pitcher_leaders(startseason = 2019, endseason = 2019)
fg_pitching_fact20 <- fg_pitcher_leaders(startseason = 2020, endseason = 2020)
fg_pitching_fact21 <- fg_pitcher_leaders(startseason = 2021, endseason = 2021)
fg_pitching_fact22 <- fg_pitcher_leaders(startseason = 2022, endseason = 2022)
fg_pitching_fact23 <- fg_pitcher_leaders(startseason = 2023, endseason = 2023)
fg_pitching_fact24 <- fg_pitcher_leaders(startseason = 2024, endseason = 2024)
fg_pitching_fact25 <- fg_pitcher_leaders(startseason = 2025, endseason = 2025)

fg_pitching_fact <- bind_rows(fg_pitching_fact15,fg_pitching_fact16,fg_pitching_fact17,fg_pitching_fact18,fg_pitching_fact19,fg_pitching_fact20,fg_pitching_fact21,fg_pitching_fact22,fg_pitching_fact23,
fg_pitching_fact24,fg_pitching_fact25)

In [49]:
write_csv(fg_pitching_fact, "fg_pitching_fact.csv")

# Hitting

---------
---------
---------

In [162]:
# Retrieve yearly FangGraphs hitting stats from 2015 to 2025

fg_hitting_fact15 <- fg_batter_leaders(startseason = 2015, endseason = 2015)
fg_hitting_fact16 <- fg_batter_leaders(startseason = 2016, endseason = 2016)
fg_hitting_fact17 <- fg_batter_leaders(startseason = 2017, endseason = 2017)
fg_hitting_fact18 <- fg_batter_leaders(startseason = 2018, endseason = 2018)
fg_hitting_fact19 <- fg_batter_leaders(startseason = 2019, endseason = 2019)
fg_hitting_fact20 <- fg_batter_leaders(startseason = 2020, endseason = 2020)
fg_hitting_fact21 <- fg_batter_leaders(startseason = 2021, endseason = 2021)
fg_hitting_fact22 <- fg_batter_leaders(startseason = 2022, endseason = 2022)
fg_hitting_fact23 <- fg_batter_leaders(startseason = 2023, endseason = 2023)
fg_hitting_fact24 <- fg_batter_leaders(startseason = 2024, endseason = 2024)
fg_hitting_fact25 <- fg_batter_leaders(startseason = 2025, endseason = 2025)


fg_hitting_fact <- bind_rows(fg_hitting_fact15,fg_hitting_fact16,fg_hitting_fact17,fg_hitting_fact18,fg_hitting_fact19,fg_hitting_fact20,fg_hitting_fact21,fg_hitting_fact22,fg_hitting_fact23,
fg_hitting_fact24,fg_hitting_fact25)

In [164]:
write_csv(fg_hitting_fact, "fg_hitting_fact.csv")

## Box Score Information for 2026 Games (Used for Live Feature)

In [3]:
mlb_schedule_2026 <- mlb_schedule(season = 2026)

In [13]:
mlb_schedule_2026 <- mlb_schedule_2026 %>% filter(date < '2026-04-23') %>% filter(date >= '2026-03-25')

In [15]:
print(mlb_schedule_2026)

── MLB Schedule data from MLB.com ─────────────────────────── baseballr 1.6.0 ──

ℹ Data updated: 2026-04-24 04:22:14 CDT



# A tibble: 371 × 67
   date      total_items total_events total_games total_games_in_progr…¹ game_pk
   <chr>           <int>        <int>       <int>                  <int>   <int>
 1 2026-03-…           1            0           1                      0  823244
 2 2026-03-…          11            0          11                      0  823649
 3 2026-03-…          11            0          11                      0  823812
 4 2026-03-…          11            0          11                      0  824704
 5 2026-03-…          11            0          11                      0  824865
 6 2026-03-…          11            0          11                      0  824541
 7 2026-03-…          11            0          11                      0  824218
 8 2026-03-…          11            0          11                      0  823325
 9 2026-03-…          11            0          11                      0  823081
10 2026-03-…          11            0          11                      0  823486
# ℹ 361

In [18]:
mlb_game_linescore(game_pk = 822745)


game_pk,home_team_id,home_team_name,away_team_id,away_team_name,num,ordinal_num,home_runs,home_hits,home_errors,⋯,away_team_record_league_record_losses,away_team_record_league_record_ties,away_team_record_league_record_pct,away_team_record_division_leader,away_team_record_wins,away_team_record_losses,away_team_record_winning_percentage,away_team_franchise_name,away_team_club_name,away_team_active
<dbl>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<int>,<int>,<int>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
822745,120,Washington Nationals,144,Atlanta Braves,1,1st,1,1,0,⋯,8,0,.692,FALSE,18,8,.692,Atlanta,Braves,TRUE
822745,120,Washington Nationals,144,Atlanta Braves,2,2nd,0,2,0,⋯,8,0,.692,FALSE,18,8,.692,Atlanta,Braves,TRUE
822745,120,Washington Nationals,144,Atlanta Braves,3,3rd,0,0,0,⋯,8,0,.692,FALSE,18,8,.692,Atlanta,Braves,TRUE
822745,120,Washington Nationals,144,Atlanta Braves,4,4th,1,2,0,⋯,8,0,.692,FALSE,18,8,.692,Atlanta,Braves,TRUE
822745,120,Washington Nationals,144,Atlanta Braves,5,5th,0,0,0,⋯,8,0,.692,FALSE,18,8,.692,Atlanta,Braves,TRUE
822745,120,Washington Nationals,144,Atlanta Braves,6,6th,0,0,0,⋯,8,0,.692,FALSE,18,8,.692,Atlanta,Braves,TRUE
822745,120,Washington Nationals,144,Atlanta Braves,7,7th,0,0,0,⋯,8,0,.692,FALSE,18,8,.692,Atlanta,Braves,TRUE
822745,120,Washington Nationals,144,Atlanta Braves,8,8th,0,0,0,⋯,8,0,.692,FALSE,18,8,.692,Atlanta,Braves,TRUE
822745,120,Washington Nationals,144,Atlanta Braves,9,9th,0,0,0,⋯,8,0,.692,FALSE,18,8,.692,Atlanta,Braves,TRUE


In [19]:
game_pks <- mlb_schedule_2026$game_pk

# 2. Initialize an empty tibble to hold the results
BoxScores26 <- tibble()

# 3. The Loop
for (i in seq_along(game_pks)) {
  current_pk <- game_pks[i]
  
  message(paste0("Fetching Box Score (", i, "/", length(game_pks), "): ", current_pk))
  
  # Fetch linescore and handle errors
  linescore_raw <- tryCatch({
    mlb_game_linescore(game_pk = current_pk)
  }, error = function(e) {
    message(paste("Error fetching game", current_pk, ":", e$message))
    return(NULL)
  })
  
  # 4. Condense the Inning-by-Inning data to a Game Summary
  if (!is.null(linescore_raw) && nrow(linescore_raw) > 0) {
    
    game_summary <- linescore_raw %>%
      summarize(
        game_pk = current_pk,
        away_team_name = first(away_team_name),
        home_team_name = first(home_team_name),
        away_runs_total = sum(away_runs, na.rm = TRUE),
        home_runs_total = sum(home_runs, na.rm = TRUE)
      )
    
    BoxScores26 <- bind_rows(BoxScores26, game_summary)
  }
  
  Sys.sleep(0.5)
}

Fetching Box Score (1/371): 823244

Fetching Box Score (2/371): 823649

Fetching Box Score (3/371): 823812

Fetching Box Score (4/371): 824704

Fetching Box Score (5/371): 824865

Fetching Box Score (6/371): 824541

Fetching Box Score (7/371): 824218

Fetching Box Score (8/371): 823325

Fetching Box Score (9/371): 823081

Fetching Box Score (10/371): 823486

Fetching Box Score (11/371): 823974

Fetching Box Score (12/371): 823163

Fetching Box Score (13/371): 823243

Fetching Box Score (14/371): 822839

Fetching Box Score (15/371): 823893

Fetching Box Score (16/371): 824946

Fetching Box Score (17/371): 824216

Fetching Box Score (18/371): 823324

Fetching Box Score (19/371): 823162

Fetching Box Score (20/371): 823971

Fetching Box Score (21/371): 823082

Fetching Box Score (22/371): 824701

Fetching Box Score (23/371): 822838

Fetching Box Score (24/371): 824862

Fetching Box Score (25/371): 823488

Fetching Box Score (26/371): 824540

Fetching Box Score (27/371): 823651

Fetching B

In [20]:
# 1. Ensure the key column types match (both should be numeric)
BoxScores26 <- BoxScores26 %>%
  mutate(game_pk = as.numeric(game_pk))

mlb_schedule_2026 <- mlb_schedule_2026 %>%
  mutate(game_pk = as.numeric(game_pk))

# 2. Join to get the date and other relevant schedule info
BoxScores26_Final <- BoxScores26 %>%
  left_join(
    mlb_schedule_2026 %>% select(game_pk, date), 
    by = "game_pk"
  )

# 3. Clean up the date format if necessary
BoxScores26_Final <- BoxScores26_Final %>%
  mutate(date = as.Date(date))

# Verification
print(head(BoxScores26_Final))

Warning message in left_join(., mlb_schedule_2026 %>% select(game_pk, date), by = "game_pk"):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 95 of `x` matches multiple rows in `y`.
ℹ Row 95 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”


# A tibble: 6 × 6
  game_pk away_team_name       home_team_name    away_runs_total home_runs_total
    <dbl> <chr>                <chr>                       <int>           <int>
1  823244 New York Yankees     San Francisco Gi…               7               0
2  823649 Pittsburgh Pirates   New York Mets                   7              11
3  823812 Chicago White Sox    Milwaukee Brewers               2              14
4  824704 Washington Nationals Chicago Cubs                   10               4
5  824865 Minnesota Twins      Baltimore Orioles               1               2
6  824541 Boston Red Sox       Cincinnati Reds                 3               0
# ℹ 1 more variable: date <date>


In [21]:
write_csv(BoxScores26_Final, "BoxScores26_Final.csv")